# Task 1: Mackey-Glass Forecasting with Paper-Aligned Baselines

This notebook revises the Task 1 study so it follows the experimental design described in
`ML_QRC_16.pdf` rather than the earlier showcase version.

The notebook mirrors the **Task 1 protocol** in the manuscript:

- Mackey-Glass benchmark with `beta=0.2`, `gamma=0.1`, `q=10`, `tau=17`
- `5000` total points with `1000 / 3000 / 1000` washout-train-test split
- one-step prediction with lookback `T = 50`
- fixed `n = 6` qubit reservoir, `F = 21` observables, `p_dep = 0.01`
- full-data (`3000`) and low-data (`300`) regimes
- Task 1 baselines from the paper:
  `ESN`, `NG-RC`, `Linear`, `Ridge`, `MLP`, `MLP-S`,
  `Ridge-PCA-d3`, `Tensor-Ridge`, and `ML-QRC-M`

The goal here is **experimental fidelity and fairness**, not forcing the exact manuscript numbers.
All QRC readout baselines use the **same cached quantum feature tensor bank**, the same split,
and the same train-only standardization protocol.


In [ ]:
%pip install -q pennylane torch numpy pandas matplotlib scikit-learn jupyter


In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.decomposition import PCA

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "scripts").exists() and (REPO_ROOT.parent / "scripts").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.paper_studies.common import (
    Standardizer,
    RegressionMLP,
    MLQRCRegressor,
    build_scalar_qrc_features,
    configure_runtime,
    count_params,
    fit_ols,
    fit_ridge_cv_regression,
    flatten_features,
    generate_mackey_glass,
    nmse,
    notebook_artifact_dir,
    predict_ols,
    predict_regressor,
    set_seed,
    tensor_project,
    tensor_ridge_project_train,
    train_regressor,
)

warnings.filterwarnings("ignore")
runtime = configure_runtime()
plt = runtime["plt"]
qml = runtime["qml"]
DEVICE = runtime["device"]
ARTIFACT_DIR = notebook_artifact_dir("task1_mg_v2")

print(f"Repo root: {REPO_ROOT}")
print(f"Artifacts: {ARTIFACT_DIR}")
print(f"PyTorch: {torch.__version__}")
print(f"Torch device: {DEVICE}")


## Design Notes

The revised notebook matches the manuscript's **Task 1** design in the following ways:

- The QRC tensor is built with **shared scalar angle encoding** `RY(pi u)` on every qubit.
- The reservoir is fixed once from `reservoir_seed = 42`.
- The readout comparison is **fair**:
  every QRC readout sees the same `T x n x F` tensor bank, derived from the same reservoir rollout.
- The neural readouts use the same optimization recipe reported in the paper:
  **Adam**, `lr = 1e-3`, `weight_decay = 1e-5`, `300` epochs, `batch_size = 32`.
- `MLP-S` is the size-matched reference used in the paper:
  **PCA-32 + one-hidden-layer MLP with hidden width 32**.
- `Ridge-PCA-d3` uses **PCA to 512 dimensions**, capped by the feasible rank in low data.
- `Tensor-Ridge` uses **HOSVD Tucker compression** with target ranks `(8, 6, 8)` at `n = 6`,
  matching the manuscript discussion.


In [ ]:
CONFIG = {
    "eval_seeds": list(range(10)),
    "reservoir_seed": 42,
    "num_points": 5000,
    "washout": 1000,
    "train_pool": 3000,
    "test_size": 1000,
    "full_train": 3000,
    "low_data_train": 300,
    "mg_beta": 0.2,
    "mg_gamma": 0.1,
    "mg_q": 10,
    "mg_tau": 17.0,
    "mg_step": 0.1,
    "mg_x0": 1.2,
    "n_qubits": 6,
    "n_layers": 1,
    "lookback": 50,
    "p_dep": 0.01,
    "qrc_batch_size": 256,
    "epochs": 300,
    "batch_size": 32,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "mlp_hidden": 64,
    "mlp_s_pca_dim": 32,
    "mlp_s_hidden": 32,
    "ridge_pca_dim": 512,
    "tensor_ranks": (8, 6, 8),
    "esn_units": 500,
    "esn_rho": 0.95,
    "esn_input_scale": 0.1,
    "esn_density": 0.1,
    "esn_ridge_alpha": 1e-4,
    "ngrc_k": 2,
    "ngrc_ridge_alpha": 1e-8,
    "ridge_alphas": (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1e3, 1e4, 1e5),
}

MLQRC_CONFIGS = [
    {"name": "ML-QRC-S", "d": 4},
    {"name": "ML-QRC-M", "d": 8},
    {"name": "ML-QRC-L", "d": 16},
    {"name": "ML-QRC-XL", "d": 32},
]

REGIMES = {
    "full": CONFIG["full_train"],
    "low_data": CONFIG["low_data_train"],
}

N_Q = CONFIG["n_qubits"]
T = CONFIG["lookback"]
F = N_Q + N_Q * (N_Q - 1) // 2
FLAT_DIM = T * N_Q * F

def mlqrc_param_count(d):
    return d * T + d * N_Q + d * F + d**3 + 1

config_view = pd.Series(CONFIG, name="value")
display(config_view.to_frame())
print(f"Observable dimension F = {F}")
print(f"Flattened QRC dimension = {FLAT_DIM:,}")


In [ ]:
flat_linear_params = FLAT_DIM + 1
flat_mlp_params = count_params(RegressionMLP(FLAT_DIM, CONFIG["mlp_hidden"]))
mlp_s_params = count_params(RegressionMLP(CONFIG["mlp_s_pca_dim"], CONFIG["mlp_s_hidden"]))

print("Reference parameter counts used in Task 1:")
print(f"  Linear / Ridge:        {flat_linear_params:>8,}")
print(f"  MLP (hidden=64):       {flat_mlp_params:>8,}")
print(f"  MLP-S (PCA-32 + 32):   {mlp_s_params:>8,}")
for cfg in MLQRC_CONFIGS:
    print(f"  {cfg['name']:12s} d={cfg['d']:>2d}: {mlqrc_param_count(cfg['d']):>8,}")


In [ ]:
mg = generate_mackey_glass(
    CONFIG["num_points"],
    CONFIG["mg_beta"],
    CONFIG["mg_gamma"],
    CONFIG["mg_q"],
    CONFIG["mg_tau"],
    CONFIG["mg_step"],
    CONFIG["mg_x0"],
)

W = CONFIG["washout"]
TR = CONFIG["train_pool"]
TE = CONFIG["test_size"]

scale_min = float(mg[W : W + TR].min())
scale_range = float(mg[W : W + TR].max() - mg[W : W + TR].min())
mg_scaled = ((mg - scale_min) / scale_range).astype(np.float32)

def unscale(values):
    values = np.asarray(values, dtype=np.float64)
    return (values * scale_range + scale_min).astype(np.float32)

decision_times = np.arange(T, len(mg), dtype=np.int32)
X_windows = np.stack([mg_scaled[idx - T : idx] for idx in decision_times]).astype(np.float32)
y_scaled = mg_scaled[decision_times].astype(np.float32)
y_original = mg[decision_times].astype(np.float32)

train_positions = np.where((decision_times >= W) & (decision_times < W + TR))[0]
test_positions = np.where((decision_times >= W + TR) & (decision_times < W + TR + TE))[0]

assert len(train_positions) == CONFIG["train_pool"]
assert len(test_positions) == CONFIG["test_size"]

print(f"Series length: {len(mg):,}")
print(f"Training windows: {len(train_positions):,}")
print(f"Test windows: {len(test_positions):,}")

fig, ax = plt.subplots(figsize=(8, 2.7))
ax.plot(mg, color="black", linewidth=0.6)
ax.axvspan(0, W, color="#d9d9d9", alpha=0.55, label="Washout")
ax.axvspan(W, W + TR, color="#a1d99b", alpha=0.35, label="Train pool")
ax.axvspan(W + TR, W + TR + TE, color="#fdae6b", alpha=0.35, label="Test")
ax.set_title("Task 1 split: Mackey-Glass trajectory")
ax.set_xlabel("Time index")
ax.set_ylabel("x(t)")
ax.legend(fontsize=7, frameon=False, ncol=3)
plt.tight_layout()
fig.savefig(ARTIFACT_DIR / "mg_series.pdf", bbox_inches="tight")
fig.savefig(ARTIFACT_DIR / "mg_series.png", bbox_inches="tight")
plt.show()


In [ ]:
qrc_cache = ARTIFACT_DIR / (
    f"qrc_feats_rseed{CONFIG['reservoir_seed']}_p{CONFIG['p_dep']:.2f}_q{N_Q}_T{T}.npy"
)

qrc_features = build_scalar_qrc_features(
    X_windows,
    qml=qml,
    n_qubits=CONFIG["n_qubits"],
    n_layers=CONFIG["n_layers"],
    t_steps=T,
    reservoir_seed=CONFIG["reservoir_seed"],
    cache_path=qrc_cache,
    batch_size=CONFIG["qrc_batch_size"],
    p_dep=CONFIG["p_dep"],
)

print(f"QRC tensor bank shape: {qrc_features.shape}")
print(f"Cached at: {qrc_cache}")
print(f"Tensor shape per sample: ({T}, {N_Q}, {F})")


In [ ]:
def fit_pca_projection(x_train_flat, target_dim, seed=0):
    effective_dim = min(target_dim, x_train_flat.shape[0] - 1, x_train_flat.shape[1])
    pca = PCA(
        n_components=effective_dim,
        svd_solver="randomized",
        random_state=seed,
    )
    pca.fit(x_train_flat.astype(np.float64))
    return pca, int(effective_dim)

def run_esn(series_scaled, seed, n_train):
    rng = np.random.default_rng(seed)

    W_res = rng.standard_normal((CONFIG["esn_units"], CONFIG["esn_units"]))
    W_res *= (rng.random((CONFIG["esn_units"], CONFIG["esn_units"])) < CONFIG["esn_density"])
    spectral_radius = np.max(np.abs(np.linalg.eigvals(W_res)))
    if spectral_radius > 1e-12:
        W_res = (CONFIG["esn_rho"] / spectral_radius) * W_res

    W_in = rng.standard_normal(CONFIG["esn_units"]) * CONFIG["esn_input_scale"]
    state = np.zeros(CONFIG["esn_units"], dtype=np.float64)
    states = np.zeros((len(series_scaled), CONFIG["esn_units"]), dtype=np.float32)

    for step, value in enumerate(series_scaled):
        state = np.tanh(W_res @ state + W_in * float(value))
        states[step] = state

    x_train = states[W : W + n_train]
    y_train = series_scaled[W + 1 : W + n_train + 1].astype(np.float64)

    gram = x_train.T.astype(np.float64) @ x_train.astype(np.float64)
    gram += CONFIG["esn_ridge_alpha"] * np.eye(CONFIG["esn_units"], dtype=np.float64)
    w_out = np.linalg.solve(gram, x_train.T.astype(np.float64) @ y_train)

    test_start = W + n_train
    x_test = states[test_start : test_start + TE]
    pred_scaled = (x_test.astype(np.float64) @ w_out).astype(np.float32)
    return nmse(unscale(series_scaled[test_start : test_start + TE]), unscale(pred_scaled))

def run_ngrc(series_scaled, n_train):
    def featurize(values, k):
        rows = []
        for step in range(k, len(values)):
            delayed = values[step - k : step].tolist()
            row = [1.0] + delayed
            for i in range(k):
                for j in range(i, k):
                    row.append(delayed[i] * delayed[j])
            rows.append(row)
        return np.asarray(rows, dtype=np.float32)

    k = CONFIG["ngrc_k"]
    train_series = series_scaled[W - k : W + n_train]
    x_train = featurize(train_series, k)
    y_train = train_series[k:].astype(np.float64)

    gram = x_train.T.astype(np.float64) @ x_train.astype(np.float64)
    gram += CONFIG["ngrc_ridge_alpha"] * np.eye(x_train.shape[1], dtype=np.float64)
    w_out = np.linalg.solve(gram, x_train.T.astype(np.float64) @ y_train)

    test_start = W + n_train
    test_series = series_scaled[test_start - k : test_start + TE]
    x_test = featurize(test_series, k)
    pred_scaled = (x_test.astype(np.float64) @ w_out).astype(np.float32)
    return nmse(unscale(series_scaled[test_start : test_start + TE]), unscale(pred_scaled))

print("Baseline helpers ready.")


In [ ]:
METHOD_ORDER = [
    "ESN",
    "NG-RC",
    "Linear",
    "Ridge",
    "MLP",
    "MLP-S",
    "Ridge-PCA-d3",
    "Tensor-Ridge",
    "ML-QRC-S",
    "ML-QRC-M",
    "ML-QRC-L",
    "ML-QRC-XL",
]

results = {regime: {method: [] for method in METHOD_ORDER} for regime in REGIMES}
detail_rows = []

t0 = time.time()
total_jobs = len(CONFIG["eval_seeds"]) * len(REGIMES)
finished = 0

for seed in CONFIG["eval_seeds"]:
    for regime, n_train in REGIMES.items():
        pos = train_positions[:n_train]

        scaler = Standardizer().fit(qrc_features[pos])
        x_train_tensor = scaler.transform(qrc_features[pos])
        x_test_tensor = scaler.transform(qrc_features[test_positions])
        x_train_flat = flatten_features(x_train_tensor)
        x_test_flat = flatten_features(x_test_tensor)
        y_train = y_scaled[pos]
        y_test = y_original[test_positions]

        score = run_esn(mg_scaled, seed=seed, n_train=n_train)
        results[regime]["ESN"].append(score)
        detail_rows.append({"seed": seed, "regime": regime, "method": "ESN", "score": score})

        score = run_ngrc(mg_scaled, n_train=n_train)
        results[regime]["NG-RC"].append(score)
        detail_rows.append({"seed": seed, "regime": regime, "method": "NG-RC", "score": score})

        ols_weights = fit_ols(x_train_flat, y_train)
        score = nmse(y_test, unscale(predict_ols(ols_weights, x_test_flat)))
        results[regime]["Linear"].append(score)
        detail_rows.append({"seed": seed, "regime": regime, "method": "Linear", "score": score})

        ridge_weights, ridge_alpha = fit_ridge_cv_regression(
            x_train_flat,
            y_train,
            CONFIG["ridge_alphas"],
        )
        score = nmse(y_test, unscale(predict_ols(ridge_weights, x_test_flat)))
        results[regime]["Ridge"].append(score)
        detail_rows.append(
            {
                "seed": seed,
                "regime": regime,
                "method": "Ridge",
                "score": score,
                "alpha": ridge_alpha,
            }
        )

        set_seed(seed)
        mlp = RegressionMLP(x_train_flat.shape[1], CONFIG["mlp_hidden"])
        mlp = train_regressor(
            mlp,
            x_train_flat,
            y_train,
            epochs=CONFIG["epochs"],
            batch_size=CONFIG["batch_size"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
        )
        score = nmse(y_test, unscale(predict_regressor(mlp, x_test_flat)))
        results[regime]["MLP"].append(score)
        detail_rows.append({"seed": seed, "regime": regime, "method": "MLP", "score": score})

        pca_small, pca_small_dim = fit_pca_projection(
            x_train_flat,
            CONFIG["mlp_s_pca_dim"],
            seed=0,
        )
        z_train_small = pca_small.transform(x_train_flat.astype(np.float64)).astype(np.float32)
        z_test_small = pca_small.transform(x_test_flat.astype(np.float64)).astype(np.float32)

        set_seed(seed)
        mlp_small = RegressionMLP(z_train_small.shape[1], CONFIG["mlp_s_hidden"])
        mlp_small = train_regressor(
            mlp_small,
            z_train_small,
            y_train,
            epochs=CONFIG["epochs"],
            batch_size=CONFIG["batch_size"],
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"],
        )
        score = nmse(y_test, unscale(predict_regressor(mlp_small, z_test_small)))
        results[regime]["MLP-S"].append(score)
        detail_rows.append(
            {
                "seed": seed,
                "regime": regime,
                "method": "MLP-S",
                "score": score,
                "pca_dim": pca_small_dim,
            }
        )

        pca_large, pca_large_dim = fit_pca_projection(
            x_train_flat,
            CONFIG["ridge_pca_dim"],
            seed=0,
        )
        z_train_large = pca_large.transform(x_train_flat.astype(np.float64)).astype(np.float32)
        z_test_large = pca_large.transform(x_test_flat.astype(np.float64)).astype(np.float32)
        ridge_pca_weights, ridge_pca_alpha = fit_ridge_cv_regression(
            z_train_large,
            y_train,
            CONFIG["ridge_alphas"],
        )
        score = nmse(y_test, unscale(predict_ols(ridge_pca_weights, z_test_large)))
        results[regime]["Ridge-PCA-d3"].append(score)
        detail_rows.append(
            {
                "seed": seed,
                "regime": regime,
                "method": "Ridge-PCA-d3",
                "score": score,
                "alpha": ridge_pca_alpha,
                "pca_dim": pca_large_dim,
            }
        )

        u_t, u_n, u_f = tensor_ridge_project_train(x_train_tensor, CONFIG["tensor_ranks"])
        core_train = flatten_features(tensor_project(x_train_tensor, u_t, u_n, u_f))
        core_test = flatten_features(tensor_project(x_test_tensor, u_t, u_n, u_f))
        tensor_weights, tensor_alpha = fit_ridge_cv_regression(
            core_train,
            y_train,
            CONFIG["ridge_alphas"],
        )
        score = nmse(y_test, unscale(predict_ols(tensor_weights, core_test)))
        results[regime]["Tensor-Ridge"].append(score)
        detail_rows.append(
            {
                "seed": seed,
                "regime": regime,
                "method": "Tensor-Ridge",
                "score": score,
                "alpha": tensor_alpha,
                "tensor_ranks": tuple(int(v) for v in (u_t.shape[1], u_n.shape[1], u_f.shape[1])),
            }
        )

        for cfg in MLQRC_CONFIGS:
            set_seed(seed)
            model = MLQRCRegressor(T, N_Q, F, cfg["d"], cfg["d"], cfg["d"])
            model = train_regressor(
                model,
                x_train_tensor,
                y_train,
                epochs=CONFIG["epochs"],
                batch_size=CONFIG["batch_size"],
                lr=CONFIG["lr"],
                weight_decay=CONFIG["weight_decay"],
            )
            score = nmse(y_test, unscale(predict_regressor(model, x_test_tensor)))
            results[regime][cfg["name"]].append(score)
            detail_rows.append(
                {
                    "seed": seed,
                    "regime": regime,
                    "method": cfg["name"],
                    "score": score,
                    "d": cfg["d"],
                }
            )

        finished += 1
        elapsed = (time.time() - t0) / 60.0
        print(
            f"[{finished:02d}/{total_jobs:02d}] "
            f"seed={seed} regime={regime:<8s} "
            f"Ridge={results[regime]['Ridge'][-1]:.4f}  "
            f"MLP-S={results[regime]['MLP-S'][-1]:.4f}  "
            f"Tensor-Ridge={results[regime]['Tensor-Ridge'][-1]:.4f}  "
            f"ML-QRC-M={results[regime]['ML-QRC-M'][-1]:.4f}  "
            f"[{elapsed:.1f} min]"
        )

detail_df = pd.DataFrame(detail_rows)
detail_csv = ARTIFACT_DIR / "task1_run_details.csv"
detail_df.to_csv(detail_csv, index=False)
print(f"Saved run-level details to: {detail_csv}")


In [ ]:
main_methods = [
    ("ESN", "ESN"),
    ("NG-RC", "NG-RC"),
    ("Linear", "Linear"),
    ("Ridge", "Ridge"),
    ("MLP", "MLP"),
    ("MLP-S", "MLP-S"),
    ("Ridge-PCA-d3", "Ridge-PCA-d3"),
    ("Tensor-Ridge", "Tensor-Ridge"),
    ("ML-QRC-M", "ML-QRC (ours)"),
]

main_rows = []
for raw_key, label in main_methods:
    row = {"Method": label, "raw_key": raw_key}
    for regime in REGIMES:
        values = np.asarray(results[regime][raw_key], dtype=np.float64)
        row[f"{regime}_mean"] = float(values.mean())
        row[f"{regime}_std"] = float(values.std())
    main_rows.append(row)

df_main = pd.DataFrame(main_rows)
display(df_main)

main_csv = ARTIFACT_DIR / "table2_main.csv"
main_csv_alias = ARTIFACT_DIR / "table1_main.csv"
results_csv = ARTIFACT_DIR / "task1_results.csv"
df_main.to_csv(main_csv, index=False, float_format="%.6f")
df_main.to_csv(main_csv_alias, index=False, float_format="%.6f")
df_main.to_csv(results_csv, index=False, float_format="%.6f")

print("Table I / main Task 1 comparison")
print(df_main.to_string(index=False))
print(f"Saved: {main_csv}")
print(f"Saved alias: {main_csv_alias}")


In [ ]:
d_rows = []

d_rows.append(
    {
        "Config": "MLP (ref.)",
        "raw_key": "MLP",
        "d": "—",
        "Params": count_params(RegressionMLP(FLAT_DIM, CONFIG["mlp_hidden"])),
        "full_mean": float(np.mean(results["full"]["MLP"])),
        "full_std": float(np.std(results["full"]["MLP"])),
        "low_data_mean": float(np.mean(results["low_data"]["MLP"])),
        "low_data_std": float(np.std(results["low_data"]["MLP"])),
    }
)

d_rows.append(
    {
        "Config": "MLP-S (ref.)",
        "raw_key": "MLP-S",
        "d": "—",
        "Params": count_params(RegressionMLP(CONFIG["mlp_s_pca_dim"], CONFIG["mlp_s_hidden"])),
        "full_mean": float(np.mean(results["full"]["MLP-S"])),
        "full_std": float(np.std(results["full"]["MLP-S"])),
        "low_data_mean": float(np.mean(results["low_data"]["MLP-S"])),
        "low_data_std": float(np.std(results["low_data"]["MLP-S"])),
    }
)

for cfg in MLQRC_CONFIGS:
    d_rows.append(
        {
            "Config": cfg["name"],
            "raw_key": cfg["name"],
            "d": cfg["d"],
            "Params": mlqrc_param_count(cfg["d"]),
            "full_mean": float(np.mean(results["full"][cfg["name"]])),
            "full_std": float(np.std(results["full"][cfg["name"]])),
            "low_data_mean": float(np.mean(results["low_data"][cfg["name"]])),
            "low_data_std": float(np.std(results["low_data"][cfg["name"]])),
        }
    )

df_d = pd.DataFrame(d_rows)
display(df_d)

d_csv = ARTIFACT_DIR / "table9_d_sweep.csv"
legacy_d_csv = ARTIFACT_DIR / "table5_ablation.csv"
df_d.to_csv(d_csv, index=False, float_format="%.6f")
df_d.to_csv(legacy_d_csv, index=False, float_format="%.6f")

print("Task 1 d-sweep")
print(df_d.to_string(index=False))
print(f"Saved: {d_csv}")
print(f"Saved compatibility copy: {legacy_d_csv}")


In [ ]:
plot_order = ["ESN", "NG-RC", "Linear", "Ridge", "MLP", "MLP-S", "Ridge-PCA-d3", "Tensor-Ridge", "ML-QRC-M"]
label_map = {
    "ESN": "ESN",
    "NG-RC": "NG-RC",
    "Linear": "Linear",
    "Ridge": "Ridge",
    "MLP": "MLP",
    "MLP-S": "MLP-S",
    "Ridge-PCA-d3": "Ridge-PCA-d3",
    "Tensor-Ridge": "Tensor-Ridge",
    "ML-QRC-M": "ML-QRC (ours)",
}
color_map = {
    "ESN": "#9ecae1",
    "NG-RC": "#6baed6",
    "Linear": "#c7e9c0",
    "Ridge": "#74c476",
    "MLP": "#bcbddc",
    "MLP-S": "#9e9ac8",
    "Ridge-PCA-d3": "#fdd0a2",
    "Tensor-Ridge": "#fd8d3c",
    "ML-QRC-M": "#005a32",
}

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.6), sharey=True)
regime_titles = {
    "full": "Full data (3000)",
    "low_data": "10% data (300)",
}

y_pos = np.arange(len(plot_order))
for ax, regime in zip(axes, ["full", "low_data"]):
    means = [np.mean(results[regime][key]) for key in plot_order]
    stds = [np.std(results[regime][key]) for key in plot_order]
    colors = [color_map[key] for key in plot_order]
    ax.barh(y_pos, means, xerr=stds, color=colors, capsize=3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([label_map[key] for key in plot_order], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("NMSE")
    ax.set_title(regime_titles[regime])
    ax.grid(True, axis="x", alpha=0.25)
    ax.grid(False, axis="y")

plt.tight_layout()
fig.savefig(ARTIFACT_DIR / "fig_tableI_bar.pdf", bbox_inches="tight")
fig.savefig(ARTIFACT_DIR / "fig_tableI_bar.png", bbox_inches="tight")
fig.savefig(ARTIFACT_DIR / "fig_tableII_bar.pdf", bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.4))
regime_titles = {
    "full": "Full data (3000)",
    "low_data": "10% data (300)",
}

mlqrc_params = [mlqrc_param_count(cfg["d"]) for cfg in MLQRC_CONFIGS]
mlqrc_full = [np.mean(results["full"][cfg["name"]]) for cfg in MLQRC_CONFIGS]
mlqrc_full_std = [np.std(results["full"][cfg["name"]]) for cfg in MLQRC_CONFIGS]
mlqrc_low = [np.mean(results["low_data"][cfg["name"]]) for cfg in MLQRC_CONFIGS]
mlqrc_low_std = [np.std(results["low_data"][cfg["name"]]) for cfg in MLQRC_CONFIGS]

ref_points = {
    "MLP": {
        "params": count_params(RegressionMLP(FLAT_DIM, CONFIG["mlp_hidden"])),
        "full": float(np.mean(results["full"]["MLP"])),
        "low": float(np.mean(results["low_data"]["MLP"])),
    },
    "MLP-S": {
        "params": count_params(RegressionMLP(CONFIG["mlp_s_pca_dim"], CONFIG["mlp_s_hidden"])),
        "full": float(np.mean(results["full"]["MLP-S"])),
        "low": float(np.mean(results["low_data"]["MLP-S"])),
    },
}

ridge_refs = {
    "full": float(np.mean(results["full"]["Ridge"])),
    "low_data": float(np.mean(results["low_data"]["Ridge"])),
}

for ax, regime in zip(axes, ["full", "low_data"]):
    means = mlqrc_full if regime == "full" else mlqrc_low
    stds = mlqrc_full_std if regime == "full" else mlqrc_low_std
    ax.errorbar(
        mlqrc_params,
        means,
        yerr=stds,
        fmt="o-",
        color="#005a32",
        capsize=3,
        linewidth=1.5,
        markersize=5,
        label="ML-QRC sweep",
    )
    for cfg, x_val, y_val in zip(MLQRC_CONFIGS, mlqrc_params, means):
        ax.annotate(cfg["name"].replace("ML-QRC-", "d="), (x_val, y_val), xytext=(4, 4), textcoords="offset points", fontsize=7)

    ax.scatter(ref_points["MLP"]["params"], ref_points["MLP"]["full" if regime == "full" else "low"], color="#756bb1", s=34, label="MLP")
    ax.scatter(ref_points["MLP-S"]["params"], ref_points["MLP-S"]["full" if regime == "full" else "low"], color="#9e9ac8", s=34, label="MLP-S")
    ax.axhline(ridge_refs[regime], color="#74c476", linestyle="--", linewidth=1.1, label="Ridge")
    ax.set_xscale("log")
    ax.set_xlabel("Trainable parameters")
    ax.set_ylabel("NMSE")
    ax.set_title(regime_titles[regime])

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.08))
plt.tight_layout()
fig.savefig(ARTIFACT_DIR / "fig_task1_d_sweep.pdf", bbox_inches="tight")
fig.savefig(ARTIFACT_DIR / "fig_task1_d_sweep.png", bbox_inches="tight")
fig.savefig(ARTIFACT_DIR / "fig_efficiency_curve.pdf", bbox_inches="tight")
plt.show()


In [ ]:
best_seed = (
    detail_df[
        (detail_df["method"] == "ML-QRC-M") & (detail_df["regime"] == "full")
    ]
    .sort_values("score")
    .iloc[0]["seed"]
)
best_seed = int(best_seed)

pos = train_positions[: CONFIG["full_train"]]
scaler = Standardizer().fit(qrc_features[pos])
x_train_tensor = scaler.transform(qrc_features[pos])
x_test_tensor = scaler.transform(qrc_features[test_positions])
x_train_flat = flatten_features(x_train_tensor)
x_test_flat = flatten_features(x_test_tensor)
y_train = y_scaled[pos]
truth = y_original[test_positions]

set_seed(best_seed)
best_mlqrc = MLQRCRegressor(T, N_Q, F, 8, 8, 8)
best_mlqrc = train_regressor(
    best_mlqrc,
    x_train_tensor,
    y_train,
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)
pred_mlqrc = unscale(predict_regressor(best_mlqrc, x_test_tensor))

ridge_weights, _ = fit_ridge_cv_regression(
    x_train_flat,
    y_train,
    CONFIG["ridge_alphas"],
)
pred_ridge = unscale(predict_ols(ridge_weights, x_test_flat))

horizon = 200
fig, ax = plt.subplots(figsize=(8.6, 3.1))
ax.plot(truth[:horizon], color="black", linewidth=1.2, label="Ground truth")
ax.plot(pred_ridge[:horizon], color="#74c476", linewidth=0.9, linestyle="--", label="QRC + Ridge")
ax.plot(pred_mlqrc[:horizon], color="#005a32", linewidth=0.9, label="QRC + ML-QRC-M")
ax.set_xlabel("Test step")
ax.set_ylabel("x(t)")
ax.set_title(f"Task 1 trajectory comparison (best ML-QRC-M seed = {best_seed})")
ax.legend(fontsize=8, frameon=False, ncol=3)
plt.tight_layout()
fig.savefig(ARTIFACT_DIR / "fig_trajectory.pdf", bbox_inches="tight")
fig.savefig(ARTIFACT_DIR / "fig_trajectory.png", bbox_inches="tight")
plt.show()


## Outputs

After execution, the notebook saves the Task 1 study artifacts under `notebooks/artifacts/task1_mg_v2/`:

- `table1_main.csv`: manuscript-aligned alias for the main Task 1 comparison
- `table2_main.csv`: compatibility copy used by existing downstream scripts
- `table9_d_sweep.csv`: Task 1 `d`-sweep with `MLP` and `MLP-S` references
- `task1_run_details.csv`: per-seed raw scores for every method and regime
- `qrc_feats_rseed42_p0.01_q6_T50.npy`: shared QRC tensor bank
- `fig_tableI_bar.pdf`: main comparison figure
- `fig_task1_d_sweep.pdf`: parameter-efficiency figure
- `fig_trajectory.pdf`: trajectory overlay for Ridge vs `ML-QRC-M`

The notebook also writes `table5_ablation.csv` and `fig_tableII_bar.pdf` as compatibility copies for
existing downstream scripts that still reference the older artifact names.
